In [91]:
import requests
import time
import json
import pandas as pd
from datetime import datetime
from pathlib import Path

In [92]:
session = requests.Session()
session.headers.update({
    'User-Agent': 'lab3-student/1.0',
    'Accept': 'application/json'
})

base_url = 'https://dummyjson.com'
log = []
folder = 'results'     
start_time = time.time()

In [93]:
def get_data(data_name, params, page, timeout=20):
    t1 = time.time()
    url = f'{base_url}/{data_name}'
    log_row = {'url': url, 'page': page, 'status': None, 'records': 0}
    
    response = session.get(url, params=params, timeout=timeout)
    
    preview = response.content[:200].decode('utf-8', errors='replace')
    if response.status_code != 200:
        log_row.update(seconds=round(time.time() - t1, 2), status=f'[ERROR]: {preview}')
        log.append(log_row)
        raise RuntimeError(f'HTTP {response.status_code}: {response.url}: {preview}')
    
    if 'application/json' not in response.headers.get('Content-Type', '').lower():
        log_row.update(seconds=round(time.time() - t1, 2), status=f'Ожидался JSON: {preview}')
        log.append(log_row)
        raise ValueError(f'Ожидался JSON: {response.url}: {preview}')
    
    items = response.json()[data_name]
    
    log_row.update(url=response.url, 
                        status=response.status_code,
                        time=round(time.time() - t1, 2),
                        records=len(items))
    log.append(log_row)
    
    return items

def collect(data_name, page_size=100, max_records=1000, max_page=10):
    try:
        result = []
        page = 1
        emptyAnswerFlag = False
        
        while (len(result) < max_records and page <= max_page and not emptyAnswerFlag):
            #print(f'[{data_name}] page #{page}')
            
            params = {'limit': min(page_size, max_records - len(result)), 'skip': len(result)}
            items = get_data(data_name=data_name, params=params, page=page)
            
            if (len(items) > 0):
                result.extend(items)
                page += 1
            else:
                emptyAnswerFlag = True
        
        print(f'[{data_name}]: DONE')
        return result
    except Exception as e:
        print(f'[{data_name}]: ERROR: {e}')


In [94]:
posts = collect('posts', page_size=20, max_records=60)
users = collect('users')
comments = collect('comments')
pd.DataFrame(log).to_csv(f'{folder}/logs/{ datetime.now().strftime("%Y_%m_%d %H.%M.%S")} collection_log.csv', index=False)

[posts]: DONE
[users]: DONE
[comments]: DONE


In [95]:
with open(f'{folder}/raw_posts.jsonl', "w", encoding="utf-8") as f:
    for rec in posts:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        
with open(f'{folder}/raw_comments.jsonl', "w", encoding="utf-8") as f:
    for rec in comments:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        
with open(f'{folder}/raw_users.jsonl', "w", encoding="utf-8") as f:
    for rec in users:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

In [96]:
posts_df = pd.json_normalize(
    posts,
    sep="_",
    record_path=None,
    meta=None,
)

posts_df.head()

,id,title,body,tags,views,userId,reactions_likes,reactions_dislikes
0,1,His mother had always taught him,His mother had always taught him not to ever t...,"[history, american, crime]",305,121,192,25
1,2,He was an expert but not in a discipline,He was an expert but not in a discipline that ...,"[french, fiction, english]",4884,91,859,32
2,3,Dave watched as the forest burned up on the hill.,Dave watched as the forest burned up on the hi...,"[magical, history, french]",4152,16,1448,39
3,4,All he wanted was a candy bar.,All he wanted was a candy bar. It didn't seem ...,"[mystery, english, american]",4548,47,359,18
4,5,Hopes and dreams were dashed that day.,Hopes and dreams were dashed that day. It shou...,"[crime, mystery, love]",626,131,119,30


In [97]:
comments_df = pd.json_normalize(
    comments,
    sep="_",
    record_path=None,
    meta=None,
)

comments_df.head()

,id,body,postId,likes,user_id,user_username,user_fullName
0,1,This is some awesome thinking!,242,3,105,emmac,Emma Wilson
1,2,What terrific math skills you're showing!,46,4,183,cameronp,Cameron Perez
2,3,You are an amazing writer!,235,2,1,emilys,Emily Johnson
3,4,Wow! You have improved so much!,31,1,89,braydenf,Brayden Fleming
4,5,Nice idea!,212,1,149,wyattp,Wyatt Perry


In [98]:
users_df = pd.json_normalize(
    users,
    sep="_",
    record_path=None,
    meta=None,
)

users_df = users_df[["id", "firstName", "lastName", "username", "email"]].copy()
users_df = users_df.rename(columns={"id": "userId"})

users_df.head()

,userId,firstName,lastName,username,email
0,1,Emily,Johnson,emilys,emily.johnson@x.dummyjson.com
1,2,Michael,Williams,michaelw,michael.williams@x.dummyjson.com
2,3,Sophia,Brown,sophiab,sophia.brown@x.dummyjson.com
3,4,James,Davis,jamesd,james.davis@x.dummyjson.com
4,5,Emma,Miller,emmaj,emma.miller@x.dummyjson.com


In [99]:
# Добавляем авторов поста к постам
posts_users_df = posts_df.copy()
posts_users_df = posts_users_df.merge(users_df, on='userId', how='left')
posts_users_df = posts_users_df.rename(columns={"firstName": "autorFirstName", "lastName": "autorLastName", 
                        "username": "autorusername", "email": "autoremail"})

print(f'Записей было: {len(posts_df)} \nЗаписей стало: {len(posts_users_df)}')

posts_users_df.head()

Записей было: 60 
Записей стало: 60


,id,title,body,tags,views,userId,reactions_likes,reactions_dislikes,autorFirstName,autorLastName,autorusername,autoremail
0,1,His mother had always taught him,His mother had always taught him not to ever t...,"[history, american, crime]",305,121,192,25,Ava,Harris,avahx,ava.harris@x.dummyjson.com
1,2,He was an expert but not in a discipline,He was an expert but not in a discipline that ...,"[french, fiction, english]",4884,91,859,32,Levi,Hicks,levih,levi.hicks@x.dummyjson.com
2,3,Dave watched as the forest burned up on the hill.,Dave watched as the forest burned up on the hi...,"[magical, history, french]",4152,16,1448,39,Avery,Perez,averyp,avery.perez@x.dummyjson.com
3,4,All he wanted was a candy bar.,All he wanted was a candy bar. It didn't seem ...,"[mystery, english, american]",4548,47,359,18,Nicholas,Bailey,nicholasb,nicholas.bailey@x.dummyjson.com
4,5,Hopes and dreams were dashed that day.,Hopes and dreams were dashed that day. It shou...,"[crime, mystery, love]",626,131,119,30,Jackson,Morales,jacksonm,jackson.morales@x.dummyjson.com


In [100]:
# Добавляем комментарии к постам
df = posts_users_df.copy()
df = df.rename(columns={"id": "postId", "body": "post_body"})
comments_df_renamed = comments_df.rename(columns={"body": "comment_body", "user_id": "commentator_id", 
                                          "user_username": "commentator_username", "id": "comment_id", "user_fullName": "commentator_fullName"}).copy()

df = df.merge(comments_df_renamed, on='postId', how='left')
print(f'Записей было: {len(posts_users_df)} \nЗаписей стало: {len(df)}')
df.head(3)

Записей было: 60 
Записей стало: 103


,postId,title,post_body,tags,views,userId,reactions_likes,reactions_dislikes,autorFirstName,autorLastName,autorusername,autoremail,comment_id,comment_body,likes,commentator_id,commentator_username,commentator_fullName
0,1,His mother had always taught him,His mother had always taught him not to ever t...,"[history, american, crime]",305,121,192,25,Ava,Harris,avahx,ava.harris@x.dummyjson.com,93.0,These are fabulous ideas!,7.0,190.0,leahw,Leah Gutierrez
1,1,His mother had always taught him,His mother had always taught him not to ever t...,"[history, american, crime]",305,121,192,25,Ava,Harris,avahx,ava.harris@x.dummyjson.com,107.0,You are a symbol of beauty.,7.0,131.0,jacksonm,Jackson Morales
2,1,His mother had always taught him,His mother had always taught him not to ever t...,"[history, american, crime]",305,121,192,25,Ava,Harris,avahx,ava.harris@x.dummyjson.com,209.0,Your eyes are like stars.,10.0,95.0,miless,Miles Stevenson


In [101]:
numbers = ['postId', 'views', 'userId', 'reactions_likes', 'reactions_dislikes',
           'comment_id', 'likes', 'commentator_id']
for column in numbers:
    df[column] = pd.to_numeric(df[column], errors='raise')

duplicates = int(df.duplicated('postId').sum())
print('Дубли по id:', duplicates)

Дубли по id: 43


Есть дубли так как на один пост может приходиться несколько комментариев и в таком случае в df будет несколько записей с одинаковым постом и разными комментариями

In [102]:
total = len(df)
comp = df.notna().sum() / total * 100
completeness = comp.reset_index()
completeness.columns = ["column", "completeness_%"]
completeness["completeness_%"] = completeness["completeness_%"].round(1)
completeness

,column,completeness_%
0,postId,100.0
1,title,100.0
2,post_body,100.0
3,tags,100.0
4,views,100.0
5,userId,100.0
6,reactions_likes,100.0
7,reactions_dislikes,100.0
8,autorFirstName,100.0
9,autorLastName,100.0


Есть пропуски в секции с информацией о комментариях, так как у некоторых постов может не быть комментариев и соответственно эта секция будет пустая

In [103]:
checks = pd.Series({
    'likes >= 0': df['reactions_likes'].ge(0).all(),
    'dislikes >= 0': df['reactions_dislikes'].ge(0).all(),
})
checks.to_frame('passed')

,passed
likes >= 0,True
dislikes >= 0,True


In [104]:
state_file = f"{folder}/state.json"

state = {}

if Path(state_file).exists():
    with open(state_file, "r", encoding="utf-8") as f:
        state = json.load(f)
        
def save_state(state: dict):
    with open(state_file, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

last_post_id = state.get("last_post_id", 0)

posts_to_save_df = df[df["postId"] > last_post_id].copy()
print(f"Новых постов: {len(posts_to_save_df)} (Предыдущий последний пост: {last_post_id})")

if len(posts_to_save_df) > 0:
    state["last_post_id"] = int(posts_to_save_df["postId"].max())
    with open(state_file, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)
    
    posts_to_save_df.to_csv(f"{folder}/res_data.csv", index=False)
    print(f"Сохранено: {f'{folder}/res_data.csv'} ({len(posts_to_save_df)} записей о постах и их комментариях)")

else:
    print("Новых записей нет")

Новых постов: 103 (Предыдущий последний пост: 0)
Сохранено: results/res_data.csv (103 записей о постах и их комментариях)


In [105]:
end_time = time.time() - start_time
print(f'Общее время работы = {end_time:.2f}с.')

Общее время работы = 2.64с.
